In [5]:
! Barnsley Fern - Fortran translation, writing a real PNG file
! Based on: https://en.wikipedia.org/wiki/Barnsley_fern
! Original: K. Moerman 2026 (BASIC dialect)
!
! Same IFS + "green channel brightening" trick as before, but instead of
! PPM this writes a genuine PNG file, hand-encoded with no external
! libraries: a minimal zlib/DEFLATE stream using uncompressed ("stored")
! blocks, plus CRC32 and Adler32 checksums implemented from scratch.

program barnsley_fern_png
  implicit none

  integer, parameter :: w = 1200, h = 1500
  integer, parameter :: hw = w / 2
  real,    parameter :: xscale = real(w) / 6.0
  real,    parameter :: yscale = real(h) / 11.0

  integer, allocatable :: img(:,:)   ! img(x,y): green channel value 0..255
  integer(kind=8) :: npoints, t
  real :: x, y, xn, r

  allocate(img(w, h))
  img = 0

  ! reproducible run, like "seed 1" in the original
  call seed_rng(1)

  x = 0.0
  y = 0.0
  npoints = int(w, 8) * int(h, 8)

  do t = 1, npoints
     call random_number(r)   ! r in [0,1)

     if (r < 0.01) then                       ! probability 1%
        xn = 0.0
        y  = 0.16 * y
     else if (r < 0.86) then                  ! probability 85%
        xn =  0.85 * x + 0.04 * y
        y  = -0.04 * x + 0.85 * y + 1.6
     else if (r < 0.93) then                  ! probability 7%
        xn = 0.20 * x - 0.26 * y
        y  = 0.23 * x + 0.22 * y + 1.6
     else                                      ! probability 7%
        xn = -0.15 * x + 0.28 * y
        y  =  0.26 * x + 0.24 * y + 0.44
     end if
     x = xn

     call plotpoint(x, y, xscale, yscale, hw, h, w, img)
  end do

  call save_png('barnsley_fern.png', img, w, h)
  print *, 'Saved barnsley_fern.png (', w, 'x', h, ')'

contains

  ! scale (x,y) to screen coords and brighten that pixel's green channel
  subroutine plotpoint(x, y, xscale, yscale, hw, h, w, img)
    real, intent(in) :: x, y, xscale, yscale
    integer, intent(in) :: hw, h, w
    integer, intent(inout) :: img(w, h)
    integer :: xscr, yscr, cgreen

    xscr = hw + nint(x * xscale)
    yscr = h  - nint(y * yscale)

    if (xscr >= 1 .and. xscr <= w .and. yscr >= 1 .and. yscr <= h) then
       cgreen = img(xscr, yscr)
       if (cgreen < 255) then
          cgreen = 7 * cgreen / 8 + 33
          if (cgreen > 255) cgreen = 255
          img(xscr, yscr) = cgreen
       end if
    end if
  end subroutine plotpoint

  ! seed the intrinsic RNG deterministically from a single integer
  subroutine seed_rng(base)
    integer, intent(in) :: base
    integer :: n, i
    integer, allocatable :: seed_arr(:)
    call random_seed(size = n)
    allocate(seed_arr(n))
    do i = 1, n
       seed_arr(i) = base * 104729 + i   ! spread a single seed across the array
    end do
    call random_seed(put = seed_arr)
  end subroutine seed_rng

  !=========================================================================
  ! Minimal PNG writer (no zlib/libpng dependency)
  !=========================================================================

  subroutine save_png(filename, img, w, h)
    character(len=*), intent(in) :: filename
    integer, intent(in) :: w, h
    integer, intent(in) :: img(w, h)
    integer :: iu
    integer(kind=8) :: crc_table(0:255)
    integer(kind=8) :: raw_size, nfull, rem, nblocks, deflate_size, idat_len
    integer(kind=8) :: crc, adler_a, adler_b, remaining_in_block, total_remaining
    integer :: i, j
    character(len=13) :: ihdr_data

    call build_crc_table(crc_table)

    open(newunit=iu, file=filename, access='stream', form='unformatted', status='replace')

    ! --- PNG signature ---------------------------------------------------
    write(iu) achar(137), achar(80), achar(78), achar(71), &
               achar(13), achar(10), achar(26), achar(10)

    ! --- IHDR chunk --------------------------------------------------------
    ihdr_data = char_be32(w) // char_be32(h) // achar(8) // achar(2) // &
                achar(0) // achar(0) // achar(0)
    call write_chunk(iu, 'IHDR', ihdr_data, 13, crc_table)

    ! --- IDAT chunk (streamed) ----------------------------------------------
    ! raw scanline data: each row = 1 filter-type byte (0 = None) + w*3 RGB bytes
    raw_size = int(h, 8) * int(1 + 3 * w, 8)
    nfull = raw_size / 65535_8
    rem   = raw_size - nfull * 65535_8
    if (rem > 0_8) then
       nblocks = nfull + 1_8
    else
       nblocks = nfull
    end if
    deflate_size = nblocks * 5_8 + raw_size          ! 5-byte stored-block header each
    idat_len = 2_8 + deflate_size + 4_8              ! zlib header(2) + deflate + adler(4)

    write(iu) char_be32_8(idat_len)
    write(iu) 'IDAT'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IDAT')

    ! zlib header: CMF=0x78, FLG=0x01 (valid check: (0x78*256+0x01) mod 31 == 0)
    call emit_byte(iu, 120, crc, crc_table)   ! 0x78
    call emit_byte(iu, 1,   crc, crc_table)   ! 0x01

    adler_a = 1_8
    adler_b = 0_8
    remaining_in_block = 0_8
    total_remaining = raw_size

    do j = 1, h
       ! filter type byte for this scanline: 0 = None
       call emit_raw_byte(iu, 0, crc, crc_table, adler_a, adler_b, &
                           remaining_in_block, total_remaining)
       do i = 1, w
          call emit_raw_byte(iu, 0,          crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)   ! R
          call emit_raw_byte(iu, img(i, j),  crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)   ! G
          call emit_raw_byte(iu, 0,          crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)   ! B
       end do
    end do

    ! Adler-32 checksum of the raw (pre-deflate) scanline data
    call emit_byte(iu, int(iand(ishft(adler_b, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_b, 255_8)),            crc, crc_table)
    call emit_byte(iu, int(iand(ishft(adler_a, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_a, 255_8)),            crc, crc_table)

    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    ! --- IEND chunk -----------------------------------------------------------
    write(iu) char_be32(0)
    write(iu) 'IEND'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IEND')
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    close(iu)
  end subroutine save_png

  ! Write one byte that is NOT part of the raw (Adler-checksummed) image
  ! data, e.g. zlib header bytes or DEFLATE block headers: CRC only.
  subroutine emit_byte(iu, byteval, crc, crc_table)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    write(iu) achar(byteval)
    crc = ieor(crc_table(iand(ieor(crc, int(byteval, 8)), 255_8)), ishft(crc, -8))
  end subroutine emit_byte

  ! Write one byte of RAW scanline data, opening a new stored-DEFLATE block
  ! whenever the previous one is full; updates both CRC and Adler-32.
  subroutine emit_raw_byte(iu, byteval, crc, crc_table, adler_a, adler_b, &
                            remaining_in_block, total_remaining)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc, adler_a, adler_b
    integer(kind=8), intent(inout) :: remaining_in_block, total_remaining
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: block_len, nlen
    logical :: is_last

    if (remaining_in_block == 0_8) then
       block_len = min(65535_8, total_remaining)
       is_last = (total_remaining <= 65535_8)
       call emit_byte(iu, merge(1, 0, is_last), crc, crc_table)      ! BFINAL+BTYPE(00)
       nlen = 65535_8 - block_len
       call emit_byte(iu, int(iand(block_len, 255_8)),            crc, crc_table)
       call emit_byte(iu, int(iand(ishft(block_len, -8), 255_8)), crc, crc_table)
       call emit_byte(iu, int(iand(nlen, 255_8)),                 crc, crc_table)
       call emit_byte(iu, int(iand(ishft(nlen, -8), 255_8)),      crc, crc_table)
       remaining_in_block = block_len
    end if

    call emit_byte(iu, byteval, crc, crc_table)
    adler_a = mod(adler_a + int(byteval, 8), 65521_8)
    adler_b = mod(adler_b + adler_a, 65521_8)

    remaining_in_block = remaining_in_block - 1_8
    total_remaining = total_remaining - 1_8
  end subroutine emit_raw_byte

  subroutine write_chunk(iu, ctype, data, dlen, crc_table)
    integer, intent(in) :: iu, dlen
    character(len=*), intent(in) :: ctype
    character(len=*), intent(in) :: data
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: crc

    write(iu) char_be32(dlen)
    write(iu) ctype
    write(iu) data(1:dlen)

    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, ctype)
    call crc_update_bytes(crc, crc_table, data(1:dlen))
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)
  end subroutine write_chunk

  subroutine crc_update_bytes(crc, crc_table, s)
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    character(len=*), intent(in) :: s
    integer :: k
    do k = 1, len(s)
      crc = ieor(crc_table(iand(ieor(crc, int(iachar(s(k:k)), 8)), 255_8)), ishft(crc, -8))
    end do
  end subroutine crc_update_bytes

  subroutine build_crc_table(crc_table)
    integer(kind=8), intent(out) :: crc_table(0:255)
    integer(kind=8), parameter :: poly = int(z'EDB88320', 8)
    integer(kind=8) :: c
    integer :: n, k
    do n = 0, 255
       c = int(n, 8)
       do k = 1, 8
          if (iand(c, 1_8) == 1_8) then
             c = ieor(ishft(c, -1), poly)
          else
             c = ishft(c, -1)
          end if
       end do
       crc_table(n) = c
    end do
  end subroutine build_crc_table

  ! 4-byte big-endian representation of a (small, non-negative) default integer
  function char_be32(v) result(s)
    integer, intent(in) :: v
    character(len=4) :: s
    integer(kind=8) :: vv
    vv = int(v, 8)
    s = achar(int(iand(ishft(vv, -24), 255_8))) // &
        achar(int(iand(ishft(vv, -16), 255_8))) // &
        achar(int(iand(ishft(vv, -8),  255_8))) // &
        achar(int(iand(vv, 255_8)))
  end function char_be32

  ! 4-byte big-endian representation of an integer(kind=8) value (masked to 32 bits)
  function char_be32_8(v) result(s)
    integer(kind=8), intent(in) :: v
    character(len=4) :: s
    s = achar(int(iand(ishft(v, -24), 255_8))) // &
        achar(int(iand(ishft(v, -16), 255_8))) // &
        achar(int(iand(ishft(v, -8),  255_8))) // &
        achar(int(iand(v, 255_8)))
  end function char_be32_8

end program barnsley_fern_png

 Saved barnsley_fern.png (        1200 x        1500 )


![Pi convergence](barnsley_fern.png)